In [2]:
import json
import os
import re
import time
from pathlib import Path
from google import genai
from google.genai import types

# ──────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────

INPUT_FILE      = Path("1959_ocr_gemini.txt")
OUTPUT_FILE     = Path("1959_ner_gemini.json")
PROGRESS_FILE   = Path("1959_ner_gemini_progress.json")   # tracks completed chunks
CHUNK_SIZE      = 50        # entries per API call — sweet spot for quality vs. cost
MODEL           = "gemini-3.1-flash-lite"                       
RETRY_ATTEMPTS  = 3
RETRY_DELAY_S   = 10        # seconds to wait before retrying a failed chunk

# ──────────────────────────────────────────────────────────────────
# SYSTEM PROMPT
# ──────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a bibliographic metadata extractor for an Italian fiscal-history
abstract journal. You will receive a raw text fragment containing a numbered sequence of
bibliography entries from the periodical "Rassegna dei Problemi Fiscali".

Your task is to locate EVERY single bibliography entry in the fragment, extract its details,
and return them as a list of JSON objects. Do NOT skip any entry, even if the data is sparse.

For each entry, extract:
- entry_id:        The number of the bibliography entry (e.g., "5882")
- title_it:        Italian title as a clean string, no markdown
- title_original:  Original title in the source language, or null
- author:          Author name(s) as a string, or null if anonymous
- journal:         Journal or newspaper name if it is an article, or null
- publisher:       Publisher name if it is a book, or null
- year:            4-digit year as a string extracted from the date, or null
- place_of_pub:    City of publication if explicitly mentioned, or null
- pages:           Number of pages, or null
- language:        Language of the original publication, or null
- keywords:        3 keywords about the topic of the entry

Rules:
- Strip ALL markdown formatting (* and **) from extracted strings.
- For "year": extract only the 4-digit year from dates like "marzo-aprile 1959".
- For "place_of_pub": do NOT infer city from journal names. Only fill if explicitly stated.
- If a field is genuinely absent, return null.
- Preserve name initials like "C.L. Harris" exactly as written.
- Return ONLY the JSON array, no prose, no markdown fences.
"""

# ──────────────────────────────────────────────────────────────────
# JSON SCHEMA
# ──────────────────────────────────────────────────────────────────

entry_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "entry_id":       types.Schema(type=types.Type.STRING),
        "title_it":       types.Schema(type=types.Type.STRING, nullable=True),
        "title_original": types.Schema(type=types.Type.STRING, nullable=True),
        "author":         types.Schema(type=types.Type.STRING, nullable=True),
        "journal":        types.Schema(type=types.Type.STRING, nullable=True),
        "publisher":      types.Schema(type=types.Type.STRING, nullable=True),
        "year":           types.Schema(type=types.Type.STRING, nullable=True),
        "place_of_pub":   types.Schema(type=types.Type.STRING, nullable=True),
        "pages":          types.Schema(type=types.Type.STRING, nullable=True),
        "language":       types.Schema(type=types.Type.STRING, nullable=True),
        "keywords":       types.Schema(type=types.Type.STRING, nullable=True),
    },
    required=["entry_id"],
)

response_schema = types.Schema(
    type=types.Type.ARRAY,
    items=entry_schema
)

# ──────────────────────────────────────────────────────────────────
# STEP 1 — SPLIT TEXT INTO ENTRY BLOCKS
# ──────────────────────────────────────────────────────────────────

def split_into_entries(text: str) -> list[tuple[str, str]]:
    """
    Split raw OCR text into individual entry blocks.

    Returns a list of (entry_id, entry_text) tuples.

    Strategy: find all positions where a new entry starts — identified by a
    standalone number (possibly bold-marked as **N** or *N*) at the beginning
    of a line, followed by text on the same or next line.  Everything between
    two such markers is one entry.
    """
    # Pattern: line starting with optional bold markers around a 4-6 digit number
    # Handles: "5882", "**5882**", "*5882*", "5882.", "5882 "
    entry_start = re.compile(
        r'(?m)^[\*\s]*(\d{4,6})[\*\.\s]',
    )

    matches = list(entry_start.finditer(text))
    if not matches:
        raise ValueError(
            "No entry boundaries detected. Check the regex against your OCR output."
        )

    entries = []
    for i, match in enumerate(matches):
        entry_id = match.group(1)
        start    = match.start()
        end      = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        entry_text = text[start:end].strip()
        entries.append((entry_id, entry_text))

    return entries


def make_chunks(entries: list[tuple[str, str]], chunk_size: int) -> list[list[tuple[str, str]]]:
    """Group entries into chunks of `chunk_size`."""
    return [entries[i:i + chunk_size] for i in range(0, len(entries), chunk_size)]


# ──────────────────────────────────────────────────────────────────
# STEP 2 — SEND ONE CHUNK TO GEMINI
# ──────────────────────────────────────────────────────────────────

def process_chunk(
    client: genai.Client,
    chunk: list[tuple[str, str]],
    chunk_index: int,
) -> list[dict]:
    """
    Send a single chunk (list of entry blocks) to Gemini and return parsed JSON.
    Retries up to RETRY_ATTEMPTS times on failure.
    """
    # Join the raw entry texts so the model sees natural context
    chunk_text = "\n\n".join(entry_text for _, entry_text in chunk)
    expected_ids = [eid for eid, _ in chunk]

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            response = client.models.generate_content(
                model=MODEL,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema=response_schema,
                ),
                contents=chunk_text,
            )

            results = json.loads(response.text.strip())

            # ── Completeness check ────────────────────────────────
            returned_ids = {r["entry_id"] for r in results}
            missing      = set(expected_ids) - returned_ids
            if missing:
                print(f"  ⚠  Chunk {chunk_index}: model skipped IDs {sorted(missing)}"
                      f" (attempt {attempt}/{RETRY_ATTEMPTS})")
                if attempt < RETRY_ATTEMPTS:
                    time.sleep(RETRY_DELAY_S)
                    continue   # retry the whole chunk

            return results

        except Exception as e:
            print(f"  ❌ Chunk {chunk_index} attempt {attempt} failed: {e}")
            if attempt < RETRY_ATTEMPTS:
                time.sleep(RETRY_DELAY_S)

    # All retries exhausted — return whatever we have (could be empty list)
    print(f"  ✗  Chunk {chunk_index}: all {RETRY_ATTEMPTS} attempts failed. Saving empty.")
    return []


# ──────────────────────────────────────────────────────────────────
# STEP 3 — LOAD / SAVE PROGRESS
# ──────────────────────────────────────────────────────────────────

def load_progress() -> dict:
    """Return {chunk_index: [list of extracted dicts]} for already-done chunks."""
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {}


def save_progress(progress: dict) -> None:
    PROGRESS_FILE.write_text(
        json.dumps(progress, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )


# ──────────────────────────────────────────────────────────────────
# STEP 4 — MERGE & DEDUPLICATE
# ──────────────────────────────────────────────────────────────────

def merge_results(progress: dict) -> list[dict]:
    """
    Flatten all chunk results into a single list, sorted by entry_id.
    Deduplicate by entry_id (last-write wins — shouldn't matter with clean data).
    """
    seen    = {}
    for chunk_idx in sorted(progress.keys(), key=int):
        for entry in progress[chunk_idx]:
            eid = entry.get("entry_id", "")
            seen[eid] = entry   # dedup: keep last occurrence

    # Sort numerically where possible
    def sort_key(eid):
        try:
            return int(eid)
        except ValueError:
            return float("inf")

    return sorted(seen.values(), key=lambda e: sort_key(e.get("entry_id", "")))


# ──────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────

def main():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise SystemExit("ERROR: set GOOGLE_API_KEY environment variable first.")

    client = genai.Client(api_key=api_key)

    # ── Read & split ──────────────────────────────────────────────
    print(f"Reading {INPUT_FILE} …")
    raw_text = INPUT_FILE.read_text(encoding="utf-8")

    entries = split_into_entries(raw_text)
    chunks  = make_chunks(entries, CHUNK_SIZE)

    print(f"  → {len(entries)} entries detected, split into {len(chunks)} chunks "
          f"of up to {CHUNK_SIZE} entries each.")

    # ── Load prior progress (allows resuming interrupted runs) ────
    progress = load_progress()
    if progress:
        print(f"  → Resuming: {len(progress)} chunks already done, "
              f"{len(chunks) - len(progress)} remaining.")

    # ── Process each chunk ────────────────────────────────────────
    for i, chunk in enumerate(chunks):
        chunk_key = str(i)   # JSON keys must be strings

        if chunk_key in progress:
            print(f"  [chunk {i+1:>4}/{len(chunks)}] skipped (already done)")
            continue

        id_range = f"{chunk[0][0]}–{chunk[-1][0]}"
        print(f"  [chunk {i+1:>4}/{len(chunks)}] entries {id_range} …", end=" ", flush=True)

        results = process_chunk(client, chunk, i)
        progress[chunk_key] = results
        save_progress(progress)   # checkpoint after every chunk

        print(f"→ {len(results)} records saved.")

        # Brief pause to stay within rate limits
        time.sleep(1)

    # ── Merge all chunks → final output ──────────────────────────
    all_entries = merge_results(progress)
    OUTPUT_FILE.write_text(
        json.dumps(all_entries, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"\n✓ Done. {len(all_entries)} total entries written to {OUTPUT_FILE}")
    print(f"  (Progress checkpoint kept at {PROGRESS_FILE} — safe to delete now.)")


if __name__ == "__main__":
    main()

Reading 1959_ocr_gemini.txt …
  → 805 entries detected, split into 17 chunks of up to 50 entries each.
  [chunk    1/17] entries 5701–5750 … 

KeyboardInterrupt: 